<a href="https://colab.research.google.com/github/xidoudou/ai-agents/blob/main/agent05_finance_assistant_reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install & Import**

In [ ]:
!pip install -q openai langgraph langchain_openai langchain_core
import re, os, requests
from google.colab import userdata

os.environ["OPENAI_API_KEY"]= userdata.get("OPENAI_API_KEY")
EXCHANGE_RATE_API_KEY = userdata.get("EXCHANGERATE_API_KEY")

**Functions & Tools**

In [ ]:
from langchain_core.tools import tool

@tool
def calculate(what):
  """Evaluate a math expression, e.g., 3 * 4 + 7"""
  return eval(what)


@tool
def get_exchange_rate(from_currency, to_currency):
  """
  Get the exchange rate from one currency to another.
  Currency codes have to be 3-letter codes like USD, EUR, GBP, etc.
  """
  API_key = EXCHANGE_RATE_API_KEY
  url = f"https://v6.exchangerate-api.com/v6/{API_key}/latest/{from_currency}"
  response = requests.get(url)
  data = response.json()
  if data['result'] != "success":
    return f"Error: could not fetch the exchange rate for {from_currency}"
  if to_currency not in data['conversion_rates']:
    return f"Error: unknown current code {to_currency}"
  return data['conversion_rates'][to_currency]


**Agent with self-reflection (Langgraph)**

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class AgentState(TypedDict):
  messages: Annotated[list[AnyMessage], operator.add]
  revisions: int
  revise: bool

class Reflection(BaseModel):
  passed: bool = Field(description= "True if draft meets all criteria")
  feedback: str = Field(description = "Specific issues to fix. Empty if passed")

class Agent:

  def __init__(self, model, tools, system = "", reflect_prompt = "", max_revisions = 3):
    self.system = system
    self.reflect_prompt = reflect_prompt
    self.max_revisions = max_revisions
    graph = StateGraph(AgentState)
    graph.add_node("llm", self.call_openai)
    graph.add_node("action", self.take_action)
    graph.add_node("reflect", self.reflect)
    graph.add_conditional_edges("llm", self.exists_actions, {True: "action", False: "reflect"})
    graph.add_edge("action", "llm")
    graph.add_conditional_edges("reflect", self.should_revise, {True: "llm", False: END})
    graph.set_entry_point("llm")
    self.graph = graph.compile()
    self.tools = {t.name: t for t in tools}
    self.model = model.bind_tools(tools)
    self.reflector = model.with_structured_output(Reflection)


  def exists_actions(self, state):
    result = state['messages'][-1]
    return len(result.tool_calls) > 0


  def call_openai(self, state):
    messages = state['messages']
    if self.system:
      messages = [SystemMessage(content = self.system)] + messages
    message = self.model.invoke(messages)
    return {'messages': [message]}

  def take_action(self, state):
    tool_calls = state['messages'][-1].tool_calls
    results = []
    for t in tool_calls:
      print (f"====== Calling: {t} ======")
      result = self.tools[t['name']].invoke(t['args'])
      results.append(ToolMessage(tool_call_id = t['id'], name = t['name'], content = str(result)))
      print("Back to the Model!")
    return {'messages': results}


  def reflect(self, state):
    # Assemble the entire conversation into a text block for the reviewer (Question + Tool Call + Tool Result + Draft)
    lines = []
    for m in state['messages']:
      line = f"[{m.type}] {m.content}"
      if getattr(m, "tool_calls", None):
        line += f" tool_calls = {m.tool_calls}"
      lines.append(line)
    transcript = "\n".join(lines)

    review = self.reflector.invoke([SystemMessage(content =self.reflect_prompt), HumanMessage(content = transcript)])
    print(f"Reflection: passed = {review.passed} | feedback = {review.feedback}")

    revisions = state.get('revisions', 0)
    revise = (not review.passed) and (revisions < self.max_revisions)
    if revise:

      feedback_msg = HumanMessage(content = f"Reviwer's Feedback on your draft:\n{review.feedback}\n Please revise your answer")
      return {"messages": [feedback_msg], "revisions": revisions + 1, "revise": True}
    return {"revise": False}

  def should_revise(self, state):
    return state['revise']


**Prompt**

In [ ]:
prompt = """
You are a helpful financial assistant.
Use the tools available to you to answer questions about currency exchange and calculations.
Only use a tool when you need to look up information or do a calculation other wise answer directly.

When giving your final answer, use plain text only (no markdown, no bold,
no bullet points). Keep it to one or two clear sentences.
""".strip()


reflect_prompt = """
You are a strict reviewer of a financial assistant's answer.
You will see the full conversation: the user's question, tool calls, tool results,
and the assistant's draft answer (the last message).

Check the draft against these criteria:
1. Every exchange rate or number in the draft comes from a tool result. Nothing invented.
2. Every amount and currency in the user's question is accounted for.
3. Arithmetic for the final answer was done with the calculate tool, and the final number matches its result.
4. Plain text only, no markdown, one or two sentences.

If all criteria pass, set passed=true and leave feedback empty.
Otherwise set passed=false and give specific, actionable feedback.
""".strip()

**Test**

In [ ]:
model = ChatOpenAI(model="gpt-6-astra")
finance_bot = Agent(model, [calculate, get_exchange_rate], system = prompt, reflect_prompt = reflect_prompt)

In [ ]:
messages = [HumanMessage(content="I have 50 euro and 30 rmb and 50 dollar, how much in total in us dollar?")]
result = finance_bot.graph.invoke({"messages": messages, "revisions": 0, "revise": False})
print(result['messages'][-1].content)

In [ ]:
for m in result['messages']:
  print(m)

In [ ]:
for m in result['messages']:
    print(f"[{m.type}] {m.content}")